# Bioinformatics Lab 2: Variant Interpretation -- Genomic Variant Analysis

**Series**: Agentic AI Science Playbook -- Bioinformatics Domain
**Goal**: Build an agent that classifies genomic variants and assesses pathogenicity using evidence-based criteria.
**Prerequisites**: BIO Lab 1, Foundation Labs 0-4.
**Time**: ~60 min.

---

## Learning Objectives

By the end of this lab, you will be able to:
- Implement ACMG/AMP variant classification using structured agent tools
- Build database-backed tools that the agent queries for evidence
- Handle the special case of novel variants (no prior classification)
- Combine multiple evidence types into a pathogenicity assessment

## Why This Matters

Clinical genomics labs classify thousands of variants per year. Each classification requires:
- Querying multiple databases (ClinVar, gnomAD, OMIM)
- Evaluating population frequency, computational predictions, and functional evidence
- Applying complex ACMG/AMP rules to arrive at a classification

An AI agent can **automate the evidence gathering and initial classification**, allowing geneticists to focus on complex cases that truly need expert judgment.

> **Clinical impact**: Novel variants (VUS) are the biggest challenge in clinical genomics. The agent's ability to synthesize multiple evidence types and produce a structured assessment helps geneticists make faster, more informed decisions.

## Prerequisites & NVIDIA SDK Connection

| Requirement | Details |
|------------|---------|
| Completed | BIO Lab 1, Foundation Labs 0-4 |
| Standards | ACMG/AMP guidelines (Richards et al., 2015) — explained inline |

**NVIDIA Connection**: **NVIDIA Parabricks** provides GPU-accelerated variant calling (DeepVariant, HaplotypeCaller) that feeds directly into the interpretation pipeline you're building. **BioNeMo** models like ESM-2 can predict variant effects on protein structure, adding another evidence type to the ACMG assessment.

## Genomics Primer: What You Need to Know (5-Minute Crash Course)

**If you're NOT a biologist**, this section gives you everything you need to follow this lab. If you ARE a biologist, feel free to skip ahead.

### DNA → Gene → Protein (The Central Dogma)

```
DNA (your genetic code)
 ↓ transcription
mRNA (a working copy)
 ↓ translation
Protein (the actual machine that does work in your cells)
```

- **DNA** is a long sequence of letters: A, C, G, T (the 4 bases)
- A **gene** is a stretch of DNA that codes for one protein
- Every 3 DNA letters (a "codon") code for 1 amino acid in the protein
- Example: `ATG` → Methionine (start), `TAA` → Stop

### What Is a Variant?

A **variant** (or mutation) is a difference in your DNA compared to the "reference" human genome. Most of us have ~4-5 million variants — the vast majority are harmless.

| Variant Type | What Changed | Effect on Protein | Example |
|-------------|-------------|------------------|---------|
| **Missense** | One letter swapped | One amino acid changes | `GAG→GTG` (Glu→Val in sickle cell) |
| **Nonsense** | Creates a premature stop | Protein is cut short | `CGA→TGA` (Arg→Stop) |
| **Frameshift** | Letters inserted/deleted | Everything after the change is garbled | `ATGCCC→ATGACCC` (insertion shifts reading) |
| **Silent** | Letter changes but amino acid stays same | No effect | `GCA→GCG` (both code for Alanine) |

### Why Does Classification Matter?

When a patient's genome is sequenced, labs find thousands of variants. The critical question is: **Is this variant causing disease, or is it harmless?**

This is what **ACMG/AMP classification** does — it's a standardized scoring system that combines multiple lines of evidence:

```
Is it rare in the population?      → More likely harmful
Does it break the protein?         → More likely harmful
Have other patients with this 
variant had the same disease?      → More likely harmful
Do computational tools predict 
it's damaging?                     → More likely harmful
```

### Key Databases

| Database | What It Contains | Website |
|---------|-----------------|---------|
| **ClinVar** | Variant classifications submitted by clinical labs | ncbi.nlm.nih.gov/clinvar |
| **gnomAD** | Population frequency (how common is this variant?) | gnomad.broadinstitute.org |
| **OMIM** | Gene-disease associations | omim.org |

### Gene Names in This Lab

| Gene | Full Name | Why It's Famous |
|------|----------|----------------|
| **BRCA1** | BReast CAncer gene 1 | Mutations increase breast/ovarian cancer risk |
| **BRCA2** | BReast CAncer gene 2 | Same — Angelina Jolie's gene |
| **TP53** | Tumor Protein p53 | "Guardian of the genome" — most mutated gene in cancer |
| **CFTR** | Cystic Fibrosis Transmembrane Regulator | Mutations cause cystic fibrosis |

> **You don't need to memorize any of this.** The tables above are reference material — come back to them as you work through the experiments. The agent does the hard work of looking up variants and applying ACMG criteria.

### How to Read Variant Notation

You'll see variants written like `BRCA1:c.5266dupC`. Here's how to decode it:

```
BRCA1:c.5266dupC
  │    │  │    │
  │    │  │    └── "dup C" = a C base was duplicated (inserted)
  │    │  └─────── "5266" = at position 5266 in the coding sequence
  │    └────────── "c." = coding DNA sequence notation
  └─────────────── "BRCA1" = the gene name
```

Common notation patterns:
- `c.5266dupC` → duplication (frameshift)
- `c.9976A>T` → A changed to T (point mutation)  
- `c.817C>T` → C changed to T (point mutation)
- `c.1521_1523delCTT` → 3 letters deleted (in-frame deletion)
- `p.Gln1756ProfsTer25` → protein effect: Glutamine at position 1756 changed to Proline, then frameshift, stop after 25 amino acids

> **Don't panic about the notation.** The agent handles all of this — you just need to know that each variant has a name that describes exactly what changed in the DNA.

Now you're ready to build the Variant Interpretation Agent!

## Background: Genomic Variant Classification

When sequencing identifies a variant in a patient's genome, clinical labs classify it using the **ACMG/AMP guidelines** (Richards et al., Genetics in Medicine, 2015):

| Class | Label | Clinical Action |
|-------|-------|----------------|
| 5 | Pathogenic (P) | Report; take clinical action |
| 4 | Likely Pathogenic (LP) | Report; consider clinical action |
| 3 | Variant of Uncertain Significance (VUS) | Report; do not act; continue research |
| 2 | Likely Benign (LB) | Typically not reported |
| 1 | Benign (B) | Not reported |

### Key Evidence Categories (ACMG)

- **Population frequency**: Rare in general population → evidence for pathogenicity
- **Functional data**: Does the variant affect protein function?
- **Computational predictions**: SIFT, PolyPhen-2, CADD scores
- **Segregation**: Does it co-segregate with disease in families?
- **Clinical**: Is this variant seen in affected individuals?

### Install Dependencies
Install the required Python packages for this lab. We need `openai` for LLM access and `pydantic` for data validation of our tool schemas.

In [ ]:
!pip install -q openai pydantic

### Connect to LLM
Set up your OpenAI or NVIDIA NIM connection. This cell detects which API you have configured and creates the client. It also imports the core libraries we will use throughout the lab.

> **Why NIM?** Data privacy, faster inference, open models. See Lab 0 for the full comparison.

In [ ]:
import os
from getpass import getpass
from openai import OpenAI
use_nim = os.environ.get("USE_NIM","").lower() in ("1","true","yes") or "NIM_API_KEY" in os.environ
if use_nim:
    if "NIM_API_KEY" not in os.environ: os.environ["NIM_API_KEY"]=getpass("NVIDIA NIM API key: ")
    client=OpenAI(base_url="https://integrate.api.nvidia.com/v1",api_key=os.environ["NIM_API_KEY"])
    MODEL=os.environ.get("NIM_MODEL","nvidia/llama-3.3-nemotron-super-49b-v1.5")
else:
    if "OPENAI_API_KEY" not in os.environ: os.environ["OPENAI_API_KEY"]=getpass("OpenAI API key: ")
    client=OpenAI()
    MODEL="gpt-4o-mini"
print(f"Using model: {MODEL}")
import json
from pydantic import BaseModel, Field
from typing import Literal

## Simulated Variant Database

In production, connect to ClinVar, gnomAD, and LOVD APIs.

### Load Simulated Variant Database
Five variants with different classifications, from the well-known pathogenic BRCA1 frameshift to a novel unclassified variant. In production, this would query ClinVar, gnomAD, and OMIM. Each entry includes population frequency, computational prediction scores, and functional evidence.

In [ ]:
VARIANT_DB = {
    "BRCA1:c.5266dupC": {
        "gene": "BRCA1", "hgvs_c": "c.5266dupC", "hgvs_p": "p.Gln1756ProfsTer25",
        "variant_type": "frameshift", "clinvar_classification": "Pathogenic",
        "population_frequency_gnomad": 0.000015,
        "computational_scores": {"sift": "deleterious", "polyphen2": "probably_damaging", "cadd_phred": 35.0},
        "functional_evidence": "loss_of_function_confirmed",
        "clinvar_submissions": 450, "associated_conditions": ["Hereditary Breast and Ovarian Cancer"],
    },
    "BRCA2:c.9976A>T": {
        "gene": "BRCA2", "hgvs_c": "c.9976A>T", "hgvs_p": "p.Lys3326Ter",
        "variant_type": "nonsense", "clinvar_classification": "Benign",
        "population_frequency_gnomad": 0.012,
        "computational_scores": {"sift": "tolerated", "polyphen2": "benign", "cadd_phred": 8.5},
        "functional_evidence": "protein_function_preserved",
        "clinvar_submissions": 85, "associated_conditions": [],
    },
    "TP53:c.817C>T": {
        "gene": "TP53", "hgvs_c": "c.817C>T", "hgvs_p": "p.Arg273Cys",
        "variant_type": "missense", "clinvar_classification": "Pathogenic",
        "population_frequency_gnomad": 0.0000012,
        "computational_scores": {"sift": "deleterious", "polyphen2": "probably_damaging", "cadd_phred": 28.3},
        "functional_evidence": "dominant_negative_confirmed",
        "clinvar_submissions": 312, "associated_conditions": ["Li-Fraumeni Syndrome", "Various Cancers"],
    },
    "CFTR:c.1521_1523delCTT": {
        "gene": "CFTR", "hgvs_c": "c.1521_1523delCTT", "hgvs_p": "p.Phe508del",
        "variant_type": "in-frame deletion", "clinvar_classification": "Pathogenic",
        "population_frequency_gnomad": 0.0206,
        "computational_scores": {"sift": "deleterious", "polyphen2": "probably_damaging", "cadd_phred": 33.7},
        "functional_evidence": "protein_misfolding_confirmed",
        "clinvar_submissions": 890, "associated_conditions": ["Cystic Fibrosis"],
    },
    "NOVEL:c.1234G>A": {
        "gene": "UNKNOWN_GENE", "hgvs_c": "c.1234G>A", "hgvs_p": "p.Val412Ile",
        "variant_type": "missense", "clinvar_classification": None,
        "population_frequency_gnomad": 0.00003,
        "computational_scores": {"sift": "deleterious", "polyphen2": "possibly_damaging", "cadd_phred": 18.5},
        "functional_evidence": "unknown",
        "clinvar_submissions": 0, "associated_conditions": [],
    },
}
print(f"Variant database loaded: {len(VARIANT_DB)} variants")

## Tool Schemas

### Define Variant Interpretation Schemas
Three tools with increasing depth: `lookup_variant` gets raw data, `classify_variant` applies ACMG rules, and `assess_pathogenicity` provides per-criterion evidence analysis. Each builds on the previous, forming an escalating analysis pipeline.

In [ ]:
class LookupVariantArgs(BaseModel):
    variant_id: str = Field(..., description="Variant identifier in format GENE:HGVS (e.g., BRCA1:c.5266dupC).")

class ClassifyVariantArgs(BaseModel):
    variant_id: str = Field(..., description="Variant identifier.")
    clinical_context: str | None = Field(None, description="Patient clinical context for classification.")

class AssessPathogenicityArgs(BaseModel):
    variant_id: str = Field(..., description="Variant identifier.")
    evidence_types: list[str] = Field(
        default_factory=lambda: ["population", "computational", "functional", "clinical"],
        description="Evidence types to include in assessment.")

OPENAI_TOOLS = [
    {"type": "function", "function": {"name": "lookup_variant", "description": "Look up a genomic variant in the database. Returns population frequency, computational scores, and ClinVar status.", "parameters": LookupVariantArgs.model_json_schema()}},
    {"type": "function", "function": {"name": "classify_variant", "description": "Classify a variant using ACMG/AMP criteria (Pathogenic/Likely Pathogenic/VUS/Likely Benign/Benign).", "parameters": ClassifyVariantArgs.model_json_schema()}},
    {"type": "function", "function": {"name": "assess_pathogenicity", "description": "Perform detailed pathogenicity assessment with evidence summary for each ACMG criterion.", "parameters": AssessPathogenicityArgs.model_json_schema()}},
]
SCHEMA_MAP = {"lookup_variant": LookupVariantArgs, "classify_variant": ClassifyVariantArgs, "assess_pathogenicity": AssessPathogenicityArgs}
print("Variant tools:", [t["function"]["name"] for t in OPENAI_TOOLS])

## Tool Implementations

### Implement Variant Tools
The lookup returns database information. The classifier uses ClinVar data for known variants and LLM reasoning for novel ones. The pathogenicity assessor evaluates population, computational, and functional evidence, mapping each to specific ACMG criteria codes (PM2, PP3, PS3, etc.).

In [ ]:
def execute_lookup_variant(args: LookupVariantArgs) -> str:
    v = VARIANT_DB.get(args.variant_id)
    if not v:
        return json.dumps({"error": f"Variant {args.variant_id!r} not found in database.", "known_variants": list(VARIANT_DB.keys())})
    return json.dumps(v, indent=2)

def execute_classify_variant(args: ClassifyVariantArgs) -> str:
    v = VARIANT_DB.get(args.variant_id)
    if not v:
        return json.dumps({"error": f"Variant {args.variant_id!r} not found."})
    if v["clinvar_classification"]:
        return json.dumps({
            "variant": args.variant_id,
            "classification": v["clinvar_classification"],
            "confidence": "high" if v["clinvar_submissions"] > 50 else "moderate",
            "source": "ClinVar",
            "n_submissions": v["clinvar_submissions"],
            "associated_conditions": v["associated_conditions"],
        }, indent=2)
    # Novel variant -- use LLM to classify
    prompt = (
        f"Classify this variant using ACMG criteria:\n{json.dumps(v)}\n"
        f"Clinical context: {args.clinical_context or 'not provided'}\n"
        "Return JSON: {\"classification\": (Pathogenic|Likely Pathogenic|VUS|Likely Benign|Benign), "
        "\"evidence_summary\": ..., \"confidence\": (high|moderate|low)}"
    )
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0, max_tokens=400,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content or "{}"

def execute_assess_pathogenicity(args: AssessPathogenicityArgs) -> str:
    v = VARIANT_DB.get(args.variant_id)
    if not v:
        return json.dumps({"error": f"Variant {args.variant_id!r} not found."})
    evidence = {}
    if "population" in args.evidence_types:
        freq = v["population_frequency_gnomad"]
        evidence["population_frequency"] = {
            "gnomad_af": freq,
            "interpretation": "Very rare (PM2 supporting)" if freq < 0.0001 else "Common in population (BA1/BS1 -- against pathogenicity)" if freq > 0.005 else "Rare",
            "criterion": "PM2" if freq < 0.0001 else "BA1" if freq > 0.005 else "BS1",
        }
    if "computational" in args.evidence_types:
        scores = v["computational_scores"]
        evidence["computational_predictions"] = {
            "sift": scores.get("sift"), "polyphen2": scores.get("polyphen2"),
            "cadd_phred": scores.get("cadd_phred"),
            "interpretation": "Damaging (PP3)" if scores.get("sift") == "deleterious" else "Benign (BP4)",
            "criterion": "PP3" if scores.get("sift") == "deleterious" else "BP4",
        }
    if "functional" in args.evidence_types:
        fev = v.get("functional_evidence", "unknown")
        evidence["functional_evidence"] = {
            "finding": fev,
            "criterion": "PS3" if "confirmed" in fev and "loss" in fev else "BS3" if "preserved" in fev else "PM1",
        }
    evidence["overall_variant_info"] = {
        "gene": v["gene"], "type": v["variant_type"],
        "clinvar": v["clinvar_classification"], "conditions": v["associated_conditions"],
    }
    return json.dumps(evidence, indent=2)

def run_tool(name, args):
    if name == "lookup_variant": return execute_lookup_variant(args)
    if name == "classify_variant": return execute_classify_variant(args)
    if name == "assess_pathogenicity": return execute_assess_pathogenicity(args)
    return f"[error] Unknown: {name}"

VAR_SYSTEM = (
    "You are a clinical genomics variant interpretation assistant trained in ACMG/AMP guidelines. "
    "Use the provided tools to look up, classify, and assess pathogenicity of genomic variants. "
    "Always note the evidence strength and limitations."
)

def var_agent(user_message):
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0,
        messages=[{"role": "system", "content": VAR_SYSTEM},
                   {"role": "user", "content": user_message}],
        tools=OPENAI_TOOLS, tool_choice="auto")
    msg = r.choices[0].message
    if not msg.tool_calls:
        return {"tool": None, "content": msg.content}
    call = msg.tool_calls[0]
    validated = SCHEMA_MAP[call.function.name](**json.loads(call.function.arguments))
    return {"tool": call.function.name, "args": validated}

## Experiments

### Experiment 1: Classify Known Variants
Test three variants: a classic BRCA1 pathogenic variant, a common benign BRCA2 variant, and a novel VUS. For known variants, the agent uses ClinVar data directly. For the novel variant, it falls back to LLM-based reasoning from the available evidence.

In [ ]:
for variant_id, label in [
    ("BRCA1:c.5266dupC", "Classic BRCA1 pathogenic"),
    ("BRCA2:c.9976A>T", "Common benign BRCA2 truncation"),
    ("NOVEL:c.1234G>A", "Novel variant (no ClinVar)"),
]:
    print(f"\n=== {label} ({variant_id}) ===")
    r = var_agent(f"Classify variant: {variant_id}")
    if r["tool"]:
        result = json.loads(run_tool(r["tool"], r["args"]))
        print(f"  Classification : {result.get('classification','?')}")
        print(f"  Confidence     : {result.get('confidence','?')}")
        print(f"  Conditions     : {result.get('associated_conditions','?')}")
        if result.get("evidence_summary"):
            print(f"  Evidence       : {result['evidence_summary'][:120]}")

<details>
<summary>Expected output (click to expand)</summary>

```
=== Classic BRCA1 pathogenic (BRCA1:c.5266dupC) ===
  Classification : Pathogenic
  Confidence     : high
  Conditions     : ['Hereditary Breast and Ovarian Cancer']

=== Common benign BRCA2 truncation (BRCA2:c.9976A>T) ===
  Classification : Benign
  Confidence     : high
  Conditions     : []

=== Novel variant (no ClinVar) (NOVEL:c.1234G>A) ===
  Classification : VUS
  Confidence     : low
  Conditions     : ?
  Evidence       : Rare in population (0.003%), computational predictions mixed...
```

> **Note**: Your output may differ slightly due to LLM non-determinism. Known variants (BRCA1, BRCA2) return deterministic ClinVar classifications. The novel variant classification depends on LLM reasoning.
</details>

### Experiment 2: Full Pathogenicity Assessment
Deep dive into CFTR deltaF508 (the most common cystic fibrosis mutation) with evidence from population, computational, and functional categories. Each ACMG criterion is evaluated individually, showing how multiple lines of evidence combine to support a pathogenic classification.

In [ ]:
# Full pathogenicity assessment for CFTR deltaF508
print("\n=== CFTR deltaF508 Full Assessment ===")
r = var_agent("Perform a full pathogenicity assessment for CFTR:c.1521_1523delCTT")
if r["tool"]:
    result = json.loads(run_tool(r["tool"], r["args"]))
    for category, data in result.items():
        if isinstance(data, dict):
            print(f"\n{category.upper()}:")
            for k, v in data.items():
                print(f"  {k}: {v}")

<details>
<summary>Expected output (click to expand)</summary>

```
=== CFTR deltaF508 Full Assessment ===

POPULATION_FREQUENCY:
  gnomad_af: 0.0206
  interpretation: Common in population (BA1/BS1 -- against pathogenicity)
  criterion: BA1

COMPUTATIONAL_PREDICTIONS:
  sift: deleterious
  polyphen2: probably_damaging
  cadd_phred: 33.7
  interpretation: Damaging (PP3)
  criterion: PP3

FUNCTIONAL_EVIDENCE:
  finding: protein_misfolding_confirmed
  criterion: PS3

OVERALL_VARIANT_INFO:
  gene: CFTR
  type: in-frame deletion
  clinvar: Pathogenic
  conditions: ['Cystic Fibrosis']
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The database values are deterministic. CFTR deltaF508 is the most common CF mutation worldwide and is classified as Pathogenic despite its relatively high population frequency (2% carrier rate).
</details>

## Reflection Questions

1. **The agent classified a known BRCA1 variant as Pathogenic.** But what if ClinVar had conflicting submissions? How should the agent handle disagreement between sources?
2. **The novel variant was classified as VUS.** What additional evidence would change this classification? How could the agent suggest next steps?
3. **How would you connect this agent** to real databases (ClinVar API, gnomAD API) instead of the simulated database? What error handling would you need?

## Summary

| Tool | Capability |
|------|-----------|
| `lookup_variant` | Database lookup: frequency, scores, ClinVar |
| `classify_variant` | ACMG classification with confidence |
| `assess_pathogenicity` | Per-criterion evidence assessment |

**Key ACMG criteria covered**:
- PM2: Very rare in population
- PP3/BP4: Computational predictions
- PS3/BS3: Functional evidence
- BA1: Common allele (against pathogenicity)

**Next**: BIO Lab 3 -- Pathway Analysis Agent for gene set enrichment and drug target identification.

---
*Agentic AI Science Playbook -- Bioinformatics Domain, Lab BIO2.*